In [3]:
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

# --- LOAD DATA ---
df = pd.read_csv('hamburg_bap_vessels_with_weather_small.csv')
df['arrival_time'] = pd.to_datetime(df['arrival_time'])
df['arrival_hours'] = (df['arrival_time'] - df['arrival_time'].min()).dt.total_seconds() / 3600.0

arrival_times = df['arrival_hours'].tolist()
handling_times = df['H_v0'].tolist()

num_vessels = len(df)
num_berths = 3
V = range(num_vessels)
B = range(num_berths)

# --- YOUR EXACT DQN WEIGHTS ---
cost_per_wait_hr = 200.0   # (c_w + c_e * e_wait)
cost_per_handle_hr = 750.0 # (c_o + c_e * e_handle)

# --- GUROBI MODEL ---
m = gp.Model("Berth_Allocation_MIP")
M = 10000

x = m.addVars(num_vessels, num_berths, vtype=GRB.BINARY, name="x")
y = m.addVars(num_vessels, num_vessels, vtype=GRB.BINARY, name="y")
t = m.addVars(num_vessels, vtype=GRB.CONTINUOUS, lb=0.0, name="t")

# EXACT MATCH TO DQN OBJECTIVE
m.setObjective(
    gp.quicksum(
        (t[v] - arrival_times[v]) * cost_per_wait_hr +
        (handling_times[v]) * cost_per_handle_hr
        for v in V
    ), GRB.MINIMIZE
)

# CONSTRAINTS
for v in V:
    m.addConstr(gp.quicksum(x[v, b] for b in B) == 1)
    m.addConstr(t[v] >= arrival_times[v])

for v in V:
    for w in V:
        if v != w:
            for b in B:
                m.addConstr(t[w] >= t[v] + handling_times[v] - M * (1 - y[v, w]) - M * (2 - x[v, b] - x[w, b]))
                m.addConstr(t[v] >= t[w] + handling_times[w] - M * y[v, w] - M * (2 - x[v, b] - x[w, b]))

m.setParam('TimeLimit', 3600)
m.optimize()

if m.status == GRB.OPTIMAL:
    print(f"\nTRUE MIP OPTIMAL COST: {m.objVal:,.2f}")

Set parameter TimeLimit to value 3600
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Non-default parameters:
TimeLimit  3600

Optimize a model with 560 rows, 140 columns and 2740 nonzeros (Min)
Model fingerprint: 0xb73c17d8
Model has 10 linear objective coefficients and an objective constant of 4.5965833333333343e+04
Variable types: 10 continuous, 130 integer (130 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+04]
  Objective range  [2e+02, 2e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 3e+04]

Presolve removed 10 rows and 10 columns
Presolve time: 0.01s
Presolved: 550 rows, 130 columns, 2730 nonzeros
Variable types: 10 continuous, 120 integer (120 binary)

Root relaxation: objective 6.488250e+04, 78 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Curr